In [1]:
from customer_ops.domain.schemas import IntentResult
from customer_ops.llm.client import MissingProviderCredentialError, configured_provider
from customer_ops.llm.experiments import evaluate_entities, evaluate_intents
from customer_ops.llm.schemas import ChatMessage

try:
    provider = configured_provider()
except MissingProviderCredentialError as error:
    raise RuntimeError(
        "No live result: configure the API key for CUSTOMER_OPS_LLM_PROVIDER and install its extra."
    ) from error
print("provider:", provider.name, "| model:", provider.model)

# Order IDs are database keys, so they must be copied verbatim, including the "order-" prefix.
system_prompt = (
    "Extract the support intent, the order ID if stated, the address if stated, and missing information. "
    "The intent must be one of: refund, cancellation, address_change, status. "
    "Copy the order ID exactly as written in the message, including any prefix such as 'order-'. "
    "Never shorten, reformat, or normalize it. Use null for anything not stated."
)
message = "My delayed order-gold-10-day has not arrived. I need a refund."
expected = IntentResult(intent="refund", order_id="order-gold-10-day")

result = provider.structured_output(
    [ChatMessage(role="system", content=system_prompt), ChatMessage(role="user", content=message)],
    IntentResult,
)
print("observed:", result.model_dump())

# Failures keep their expected and observed values, so a dropped prefix shows up here.
cases = [("order-id-is-preserved", expected, result)]
{
    "intent": evaluate_intents(cases).as_dict(),
    "entities": evaluate_entities(cases).as_dict(),
}

C:\Users\QTF4584\Private\customerops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


provider: huggingface | model: Qwen/Qwen2.5-1.5B-Instruct


observed: {'intent': 'refund', 'order_id': 'gold-10-day', 'proposed_address': None, 'missing_information': ['address']}


{'intent': {'metric': 'intent_accuracy',
  'correct': 1,
  'total': 1,
  'failures': [],
  'rate': 1.0},
 'entities': {'metric': 'entity_accuracy',
  'correct': 0,
  'total': 1,
  'failures': [{'case_id': 'order-id-is-preserved',
    'expected': {'order_id': 'order-gold-10-day', 'proposed_address': None},
    'observed': {'order_id': 'gold-10-day', 'proposed_address': None}}],
  'rate': 0.0}}

In [6]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Optional
import os

class Ingredient(BaseModel):
    name: str = Field(description="Name of the ingredient.")
    quantity: str = Field(description="Quantity of the ingredient, including units.")

class Recipe(BaseModel):
    recipe_name: str = Field(description="The name of the recipe.")
    prep_time_minutes: Optional[int] = Field(description="Optional time in minutes to prepare the recipe.")
    ingredients: List[Ingredient]
    instructions: List[str]

client = genai.Client(api_key=os.getenv("CUSTOMER_OPS_GEMINI_API_KEY"))

prompt = """
Please extract the recipe from the following text.
The user wants to make delicious chocolate chip cookies.
They need 2 and 1/4 cups of all-purpose flour, 1 teaspoon of baking soda,
1 teaspoon of salt, 1 cup of unsalted butter (softened), 3/4 cup of granulated sugar,
3/4 cup of packed brown sugar, 1 teaspoon of vanilla extract, and 2 large eggs.
For the best part, they'll need 2 cups of semisweet chocolate chips.
First, preheat the oven to 375°F (190°C). Then, in a small bowl, whisk together the flour,
baking soda, and salt. In a large bowl, cream together the butter, granulated sugar, and brown sugar
until light and fluffy. Beat in the vanilla and eggs, one at a time. Gradually beat in the dry
ingredients until just combined. Finally, stir in the chocolate chips. Drop by rounded tablespoons
onto ungreased baking sheets and bake for 9 to 11 minutes.
"""

interaction = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=None,
            response_mime_type="application/json",
            response_schema=Recipe.model_json_schema(),
        ),
    )

print("Raw response: ", interaction.text)
recipe = Recipe.model_validate_json(interaction.text)
print(recipe)

Raw response:  {
  "recipe_name": "Delicious Chocolate Chip Cookies",
  "prep_time_minutes": null,
  "ingredients": [
    {
      "name": "all-purpose flour",
      "quantity": "2 and 1/4 cups"
    },
    {
      "name": "baking soda",
      "quantity": "1 teaspoon"
    },
    {
      "name": "salt",
      "quantity": "1 teaspoon"
    },
    {
      "name": "unsalted butter",
      "quantity": "1 cup"
    },
    {
      "name": "granulated sugar",
      "quantity": "3/4 cup"
    },
    {
      "name": "packed brown sugar",
      "quantity": "3/4 cup"
    },
    {
      "name": "vanilla extract",
      "quantity": "1 teaspoon"
    },
    {
      "name": "large eggs",
      "quantity": "2"
    },
    {
      "name": "semisweet chocolate chips",
      "quantity": "2 cups"
    }
  ],
  "instructions": [
    "Preheat the oven to 375°F (190°C).",
    "In a small bowl, whisk together the flour, baking soda, and salt.",
    "In a large bowl, cream together the butter, granulated sugar, and bro